In [1]:
from pathlib import Path

ROOT = Path(r"D:\ML\ML_Research_Radar").resolve()

# Показывать папку, но не показывать файлы внутри
HIDE_FILES_SUBTREES = [
    ROOT / "data",
]

# Показывать только саму папку одной строкой и не раскрывать её
COLLAPSE_SUBTREES = [
    ROOT / "archive",
    ROOT / ".git" / "objects",
]

# Полностью исключить из дерева
EXCLUDE_NAMES = {
    "__pycache__",
    ".pytest_cache",
    ".ipynb_checkpoints",
}

HIDE_FILES_SUBTREES = [p.resolve() for p in HIDE_FILES_SUBTREES if p.exists()]
COLLAPSE_SUBTREES = [p.resolve() for p in COLLAPSE_SUBTREES if p.exists()]

def is_under(path: Path, parent: Path) -> bool:
    try:
        path = path.resolve()
        parent = parent.resolve()
        return path == parent or parent in path.parents
    except Exception:
        return False

def should_hide_files(path: Path) -> bool:
    return any(is_under(path, subtree) for subtree in HIDE_FILES_SUBTREES)

def should_collapse(path: Path) -> bool:
    return any(path.resolve() == subtree for subtree in COLLAPSE_SUBTREES)

def count_dir(path: Path):
    n_dirs = 0
    n_files = 0
    try:
        for p in path.iterdir():
            if p.is_dir():
                n_dirs += 1
            else:
                n_files += 1
    except PermissionError:
        return None, None
    return n_dirs, n_files

def print_tree(path: Path, prefix: str = "", max_depth: int = 8, depth: int = 0):
    if depth > max_depth:
        return

    try:
        items = list(path.iterdir())
    except PermissionError:
        print(prefix + "└── [PermissionError]")
        return

    items = [p for p in items if p.name not in EXCLUDE_NAMES]

    hide_files_here = should_hide_files(path)

    if hide_files_here:
        items = [p for p in items if p.is_dir()]

    items = sorted(items, key=lambda p: (p.is_file(), p.name.lower()))

    for i, item in enumerate(items):
        is_last = i == len(items) - 1
        connector = "└── " if is_last else "├── "
        line = prefix + connector + item.name

        if item.is_dir():
            n_dirs, n_files = count_dir(item)

            if should_collapse(item):
                if n_dirs is not None:
                    line += f"  [collapsed: {n_files} files, {n_dirs} dirs]"
                print(line)
                continue

            if should_hide_files(item):
                if n_dirs is not None:
                    line += f"  [files hidden: {n_files}, dirs: {n_dirs}]"

        print(line)

        if item.is_dir():
            extension = "    " if is_last else "│   "
            print_tree(item, prefix + extension, max_depth=max_depth, depth=depth + 1)

hidden_note_parts = []
if HIDE_FILES_SUBTREES:
    hidden_note_parts.append(
        "files hidden under: " + ", ".join(str(p.relative_to(ROOT)) for p in HIDE_FILES_SUBTREES)
    )
if COLLAPSE_SUBTREES:
    hidden_note_parts.append(
        "collapsed: " + ", ".join(str(p.relative_to(ROOT)) for p in COLLAPSE_SUBTREES)
    )

hidden_note = f"  [{' | '.join(hidden_note_parts)}]" if hidden_note_parts else ""

print(f"\n📁 Project tree for: {ROOT}\n")
print(ROOT.name + hidden_note)
print_tree(ROOT, max_depth=20)


📁 Project tree for: D:\ML\ML_Research_Radar

ML_Research_Radar  [files hidden under: data | collapsed: archive, .git\objects]
├── .git
│   ├── hooks
│   │   ├── applypatch-msg.sample
│   │   ├── commit-msg.sample
│   │   ├── fsmonitor-watchman.sample
│   │   ├── post-update.sample
│   │   ├── pre-applypatch.sample
│   │   ├── pre-commit.sample
│   │   ├── pre-merge-commit.sample
│   │   ├── pre-push.sample
│   │   ├── pre-rebase.sample
│   │   ├── pre-receive.sample
│   │   ├── prepare-commit-msg.sample
│   │   ├── push-to-checkout.sample
│   │   ├── sendemail-validate.sample
│   │   └── update.sample
│   ├── info
│   │   └── exclude
│   ├── logs
│   │   ├── refs
│   │   │   ├── heads
│   │   │   │   └── main
│   │   │   └── remotes
│   │   │       └── origin
│   │   │           └── main
│   │   └── HEAD
│   ├── objects  [collapsed: 0 files, 240 dirs]
│   ├── refs
│   │   ├── heads
│   │   │   └── main
│   │   ├── remotes
│   │   │   └── origin
│   │   │       └── main
│   │   └── tag

In [1]:
# Purpose: print project tree, but hide ALL files anywhere under ROOT/data (show folders only)

from pathlib import Path

ROOT = Path(r"D:\ML\ML_Research_Radar").resolve()
DATA_ROOT = (ROOT / "data").resolve()   # everything under this subtree will hide files

def is_under(path: Path, parent: Path) -> bool:
    try:
        path = path.resolve()
        parent = parent.resolve()
        return parent in path.parents or path == parent
    except Exception:
        return False

def count_dir(path: Path):
    n_dirs = 0
    n_files = 0
    try:
        for p in path.iterdir():
            if p.is_dir():
                n_dirs += 1
            else:
                n_files += 1
    except PermissionError:
        return None, None
    return n_dirs, n_files

def print_tree(path: Path, prefix: str = "", max_depth: int = 8, depth: int = 0):
    if depth > max_depth:
        return

    try:
        items = list(path.iterdir())
    except PermissionError:
        print(prefix + "└── [PermissionError]")
        return

    hide_files_here = is_under(path, DATA_ROOT)

    if hide_files_here:
        # show only directories
        items = sorted([p for p in items if p.is_dir()], key=lambda p: p.name.lower())
    else:
        items = sorted(items, key=lambda p: (p.is_file(), p.name.lower()))

    for i, item in enumerate(items):
        is_last = i == len(items) - 1
        connector = "└── " if is_last else "├── "
        line = prefix + connector + item.name

        # Add a note for directories under data to indicate file counts are hidden
        if item.is_dir() and is_under(item, DATA_ROOT):
            n_dirs, n_files = count_dir(item)
            if n_dirs is not None:
                line += f"  [files hidden: {n_files}, dirs: {n_dirs}]"

        print(line)

        if item.is_dir():
            extension = "    " if is_last else "│   "
            print_tree(item, prefix + extension, max_depth=max_depth, depth=depth + 1)

print(f"\n📁 Project tree for: {ROOT}\n")
print(ROOT.name + ("  [data files hidden]" if DATA_ROOT.exists() else ""))

print_tree(ROOT, max_depth=20)


📁 Project tree for: D:\ML\ML_Research_Radar

ML_Research_Radar  [data files hidden]
├── .git
│   ├── hooks
│   │   ├── applypatch-msg.sample
│   │   ├── commit-msg.sample
│   │   ├── fsmonitor-watchman.sample
│   │   ├── post-update.sample
│   │   ├── pre-applypatch.sample
│   │   ├── pre-commit.sample
│   │   ├── pre-merge-commit.sample
│   │   ├── pre-push.sample
│   │   ├── pre-rebase.sample
│   │   ├── pre-receive.sample
│   │   ├── prepare-commit-msg.sample
│   │   ├── push-to-checkout.sample
│   │   ├── sendemail-validate.sample
│   │   └── update.sample
│   ├── info
│   │   └── exclude
│   ├── logs
│   │   ├── refs
│   │   │   ├── heads
│   │   │   │   └── main
│   │   │   └── remotes
│   │   │       └── origin
│   │   │           └── main
│   │   └── HEAD
│   ├── objects
│   │   ├── 00
│   │   │   ├── 8807012d52255397833fd5927ca48fe90d7620
│   │   │   └── 8d4860139c350ac9548238a04af1f8f8879d70
│   │   ├── 01
│   │   │   └── d81599e7bd349ceec63423cc9290d075d20073
│   │   ├── 07

In [1]:
import json

path = "D:/ML/ML_Research_Radar/data/analytics/reconciled/canonical_documents.jsonl"

with open(path, "r", encoding="utf-8") as f:
    docs = [json.loads(line) for line in f]

# multi-source docs
multi = [d for d in docs if d.get("source_count", 1) > 1]

print("multi-source docs:", len(multi))

for d in multi[:5]:
    print("\n---")
    print("title:", d.get("title"))
    print("sources:", d.get("source_ids"))
    print("doc_ids:", d.get("doc_ids"))
    print("journal:", d.get("journal"))
    print("publication_type:", d.get("publication_type"))

multi-source docs: 85

---
title: Sentiment trading with large language models
sources: {'arxiv': '2412.19245v1', 'openalex': 'https://openalex.org/W4392859372', 'semantic_scholar': 'f0f339d02a94d9609fc30561e72b3fe1ad83bca4'}
doc_ids: ['272a9a8e9fede9bd5148a8883f65d9a2', 'bc0ac931a17e03106ccc4ea61c897425']
journal: Finance research letters
publication_type: article

---
title: Visual style prompt learning using diffusion models for blind face restoration
sources: {'arxiv': '2412.21042v1', 'openalex': 'https://openalex.org/W4405865448', 'semantic_scholar': '88121dbcf35cc64bb2be40be954ff4ae1ee32ff0'}
doc_ids: ['3762b4eefec42380d9b6ff508041cd4b', '441ff4bad2d60a53d2da7dc18ec43b64']
journal: Pattern Recognition
publication_type: article

---
title: YOLO-MST: Multiscale deep learning method for infrared small target detection based on super-resolution and YOLO
sources: {'arxiv': '2412.19878v1', 'openalex': 'https://openalex.org/W4408992192', 'semantic_scholar': 'fc3adb3d69985d850a3bfda02518